# Cuadernillo 02: Dimensiones de la Calidad de los Datos (Auditoría)

Antes de limpiar un dataset, necesitamos auditarlo. Según las buenas prácticas (y como bien resume nuestro mapa mental de anomalías), primero debemos **Detectar** los problemas antes de aplicar un **Tratamiento**. 

Para evaluar si nuestros datos son de calidad, los pasamos por 5 filtros o "dimensiones":
1. **Unicidad:** ¿Hay registros duplicados que inflen mis números?
2. **Completitud:** ¿Tengo todos los datos o hay vacíos (ocultos o explícitos)?
3. **Validez:** ¿El dato respeta las reglas de su propia columna (tipo, rango, categoría)?
4. **Consistencia:** ¿El dato tiene sentido cuando lo cruzo con otra columna?
5. **Exactitud:** El dato existe y es válido, pero... ¿es la verdad?

*Preparación: Vamos a crear un DataFrame escolar con errores a propósito para auditarlo.*

In [1]:
import pandas as pd
import numpy as np

# Creamos un dataset escolar ficticio, inyectando errores intencionales
datos = {
    'id': ['001', '002', '003', '004', '005', '006', '007', '007'], # 007 repetido
    'curso': ['5ºA', '5ºB', '5ºA', '5ºC', '5ºA', '5ºB', '5ºA', '5ºA'], # 5ºC no debería existir
    'nota_txt': ['8', '10', 'sin dato', '11', '7', '', '9', '9'], # 11 fuera de rango, textos infiltrados
    'cantidad_materias': [3, 4, 3, 5, 2, 4, 3, 3],
    'costo_materia': [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000],
    'total_pagado': [3000, 4000, 3000, 5000, 9999, 4000, 3000, 3000] # 9999 es inconsistente matemáticamente
}

escolar = pd.DataFrame(datos)
display(escolar)

,id,curso,nota_txt,cantidad_materias,costo_materia,total_pagado
0,001,5ºA,8,3,1000,3000
1,002,5ºB,10,4,1000,4000
2,003,5ºA,sin dato,3,1000,3000
3,004,5ºC,11,5,1000,5000
4,005,5ºA,7,2,1000,9999
5,006,5ºB,,4,1000,4000
6,007,5ºA,9,3,1000,3000
7,007,5ºA,9,3,1000,3000


## 1. Unicidad (Uniqueness)
La unicidad busca responder a la pregunta: ¿Cada entidad de mi mundo real está representada por una y solo una fila en mi tabla?

Usamos el método `.duplicated()` para encontrar repeticiones. Al sumarlo con `.sum()`, obtenemos la cantidad de filas que son copias exactas de una fila anterior.

In [2]:
# 1. Contamos cuántos duplicados exactos hay
cantidad_duplicados = escolar.duplicated().sum()
print(f"Filas repetidas encontradas: {cantidad_duplicados}")

# 2. Vemos cuáles son esas filas duplicadas (usando el método duplicated como filtro)
# keep=False nos muestra tanto el original como la copia para ver el grupo entero
filas_repetidas = escolar[escolar.duplicated(keep=False)]

print("\nDetalle de los registros duplicados (ID 007 aparece dos veces):")
display(filas_repetidas)

Filas repetidas encontradas: 1

Detalle de los registros duplicados (ID 007 aparece dos veces):


,id,curso,nota_txt,cantidad_materias,costo_materia,total_pagado
6,007,5ºA,9,3,1000,3000
7,007,5ºA,9,3,1000,3000


## 2. Completitud (Completeness) y Etiquetas de Ausencia
¿Están todos los datos que deberían estar?
El método básico `isna().sum()` busca nulos reales (NaN). Sin embargo, en la vida real, la ausencia de datos a veces viene disfrazada como cadenas de texto ("sin dato", "N/A" o espacios en blanco).

Para calcular la completitud real, debemos normalizar el texto y buscar tanto los nulos como estas etiquetas trampa. Luego, podemos usar `.mean()` para calcular qué porcentaje de nuestra columna está realmente completa.

In [3]:
# Normalizamos la columna: forzamos a texto y limpiamos espacios vacíos en los bordes
nota_texto = escolar['nota_txt'].astype('string').str.strip()

# Buscamos nulos reales (isna) O (|) textos que sabemos que significan ausencia (isin)
mascara_ausentes = nota_texto.isna() | nota_texto.isin(["", "sin dato"])

print("Registros con notas ausentes (completitud fallida):")
display(escolar[mascara_ausentes])

# Calculamos el porcentaje de completitud. 
# Si invertimos la máscara con (~), tenemos los presentes. .mean() calcula el porcentaje (0 a 1)
porcentaje_completitud = (~mascara_ausentes).mean() * 100
print(f"\nPorcentaje de notas válidamente completas: {porcentaje_completitud:.1f}%")

Registros con notas ausentes (completitud fallida):


,id,curso,nota_txt,cantidad_materias,costo_materia,total_pagado
2,003,5ºA,sin dato,3,1000,3000
5,006,5ºB,,4,1000,4000



Porcentaje de notas válidamente completas: 75.0%


## 3. Validez (Validity)
Un dato puede estar presente, pero no ser válido. La validez verifica que el dato respete las reglas de formato, rango o categoría de negocio.

**A. Validez de Categoría:** Un curso solo puede ser '5ºA' o '5ºB'.
**B. Validez de Rango:** Una nota debe ser un número entre 1 y 10. Si intentamos convertir la columna a números, usamos `errors='coerce'` para que los textos ("sin dato") no rompan el programa, sino que se conviertan temporalmente en nulos y nos dejen evaluar la matemática.

In [4]:
# A. VALIDEZ DE CATEGORÍA
cursos_permitidos = ['5ºA', '5ºB']
mascara_curso_invalido = ~escolar['curso'].isin(cursos_permitidos)

print("--- Control de Categorías ---")
print("Estudiantes en cursos que no existen:")
display(escolar[mascara_curso_invalido])

# B. VALIDEZ DE RANGO (con conversión auxiliar)
# errors='coerce' fuerza los textos a NaN para no interrumpir el proceso
nota_num = pd.to_numeric(nota_texto, errors='coerce')

# Buscamos los que SÍ se pudieron convertir a número (notna) PERO (&) que NO (~) están entre 1 y 10
mascara_fuera_rango = nota_num.notna() & ~nota_num.between(1, 10)

print("\n--- Control de Rangos ---")
print("Estudiantes con notas fuera del rango permitido (1 al 10):")
display(escolar[mascara_fuera_rango])

--- Control de Categorías ---
Estudiantes en cursos que no existen:


,id,curso,nota_txt,cantidad_materias,costo_materia,total_pagado
3,004,5ºC,11,5,1000,5000



--- Control de Rangos ---
Estudiantes con notas fuera del rango permitido (1 al 10):


,id,curso,nota_txt,cantidad_materias,costo_materia,total_pagado
3,004,5ºC,11,5,1000,5000


## 4. Consistencia (Consistency)
La consistencia implica comparar dos o más variables dentro de la misma fila para ver si tienen sentido lógico juntas. 

En nuestro caso de negocio, la regla matemática estricta es que el `total_pagado` debe ser exactamente igual a `cantidad_materias` multiplicado por `costo_materia`. Creamos una columna temporal que evalúe si esto coincide (`True`) o no (`False`).

In [5]:
# Creamos una columna booleana que evalúa la regla de consistencia
escolar['coincide_pago'] = escolar['total_pagado'] == (escolar['cantidad_materias'] * escolar['costo_materia'])

print("Auditoría de Consistencia Financiera:")
# Filtramos solo las columnas que nos interesan para ver el problema
display(escolar[['id', 'cantidad_materias', 'costo_materia', 'total_pagado', 'coincide_pago']])

# Aislamos el error
print("\nFilas inconsistentes detectadas:")
display(escolar[~escolar['coincide_pago']])

Auditoría de Consistencia Financiera:


,id,cantidad_materias,costo_materia,total_pagado,coincide_pago
0,001,3,1000,3000,True
1,002,4,1000,4000,True
2,003,3,1000,3000,True
3,004,5,1000,5000,True
4,005,2,1000,9999,False
5,006,4,1000,4000,True
6,007,3,1000,3000,True
7,007,3,1000,3000,True



Filas inconsistentes detectadas:


,id,curso,nota_txt,cantidad_materias,costo_materia,total_pagado,coincide_pago
4,005,5ºA,7,2,1000,9999,False
